In [1]:
# Cell 1) 라이브러리 / DB 설정
import pandas as pd
import numpy as np
import re
import urllib.parse
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SRC_SCHEMA = "d1_machine_log"
SRC_TABLES = ["Vision1_machine_log", "Vision2_machine_log"]

TGT_SCHEMA = "d1_machine_log"
TGT_TABLE  = "vision_op_gap"

def get_engine(cfg=DB_CONFIG):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine()

In [2]:
# Cell 2) 타겟 스키마/테이블 생성 (없으면 생성)
ddl = f"""
CREATE SCHEMA IF NOT EXISTS {TGT_SCHEMA};

CREATE TABLE IF NOT EXISTS {TGT_SCHEMA}.{TGT_TABLE} (
    id        BIGSERIAL PRIMARY KEY,
    end_day   TEXT NOT NULL,
    station   TEXT NOT NULL,
    end_time  TEXT NOT NULL,
    contents  TEXT NOT NULL,
    remark    TEXT NOT NULL,
    op_gap    DOUBLE PRECISION NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl))

print(f"[OK] ensured {TGT_SCHEMA}.{TGT_TABLE}")

[OK] ensured d1_machine_log.vision_op_gap


In [3]:
# Cell 3) 소스 테이블에서 필터 조건에 맞는 데이터 로딩 (대문자 테이블명 대응)

pattern_sql = r"^BA1WJ.{0,26}\s::\sTEST\sRESULT\s::\s(OK|NG)$"  # BA1WJ(최대 31자) :: TEST RESULT :: OK/NG

def q_ident(name: str) -> str:
    # 대문자/특수문자 있는 식별자 안전하게 " " 처리
    return '"' + name.replace('"', '""') + '"'

schema_q = q_ident(SRC_SCHEMA)

union_sql = "\nUNION ALL\n".join([
f"""
SELECT
    end_day::text AS end_day,
    station::text AS station,
    end_time::text AS end_time,
    contents::text AS contents
FROM {schema_q}.{q_ident(t)}
WHERE contents ~ :pattern
"""
for t in SRC_TABLES])

sql = f"{union_sql}"

df = pd.read_sql_query(text(sql), engine, params={"pattern": pattern_sql})
print(f"[OK] loaded rows = {len(df):,}")
df.head()

[OK] loaded rows = 219,629


,end_day,station,end_time,contents
0,20251001,Vision1,01:25:33.71,BA1WJ25273504257SJ8T-14F014-AE :: TEST RESULT ...
1,20251001,Vision1,01:25:58.21,BA1WJ25273504255SJ8T-14F014-AE :: TEST RESULT ...
2,20251001,Vision1,01:26:12.13,BA1WJ25273504254SJ8T-14F014-AE :: TEST RESULT ...
3,20251001,Vision1,01:26:39.74,BA1WJ25273504253SJ8T-14F014-AE :: TEST RESULT ...
4,20251001,Vision1,01:26:55.06,BA1WJ25273504252SJ8T-14F014-AE :: TEST RESULT ...


In [4]:
# Cell 4) contents에서 바코드 추출 + remark 생성
# - 바코드 = contents의 첫 토큰 (공백 전까지)
# - 18번째 문자(1-index) == 인덱스 17 (0-index)
# - J 또는 S 아니면 Non-PD, 아니면 PD

def extract_barcode(contents: str) -> str:
    if contents is None:
        return ""
    # "BA1WJ... :: TEST RESULT :: OK" 에서 첫 토큰(공백 전)을 바코드로 사용
    return str(contents).split(" ")[0].strip()

df["barcode"] = df["contents"].map(extract_barcode)

def make_remark(barcode: str) -> str:
    s = str(barcode)
    # 18번째 문자가 없으면 Non-PD로 처리
    if len(s) < 18:
        return "Non-PD"
    ch = s[17]  # 18th char (1-indexed)
    return "PD" if ch in ("J", "S") else "Non-PD"

df["remark"] = df["barcode"].map(make_remark)

# barcode 컬럼은 최종 저장 스펙에는 없으므로 필요하면 확인용으로만 사용
df[["end_day","station","end_time","contents","barcode","remark"]].head()

,end_day,station,end_time,contents,barcode,remark
0,20251001,Vision1,01:25:33.71,BA1WJ25273504257SJ8T-14F014-AE :: TEST RESULT ...,BA1WJ25273504257SJ8T-14F014-AE,PD
1,20251001,Vision1,01:25:58.21,BA1WJ25273504255SJ8T-14F014-AE :: TEST RESULT ...,BA1WJ25273504255SJ8T-14F014-AE,PD
2,20251001,Vision1,01:26:12.13,BA1WJ25273504254SJ8T-14F014-AE :: TEST RESULT ...,BA1WJ25273504254SJ8T-14F014-AE,PD
3,20251001,Vision1,01:26:39.74,BA1WJ25273504253SJ8T-14F014-AE :: TEST RESULT ...,BA1WJ25273504253SJ8T-14F014-AE,PD
4,20251001,Vision1,01:26:55.06,BA1WJ25273504252SJ8T-14F014-AE :: TEST RESULT ...,BA1WJ25273504252SJ8T-14F014-AE,PD


In [5]:
# Cell 5) 정렬 + op_gap 계산
# - end_time 오름차순 (요청)
# - 실제 gap 계산을 위해 end_day + end_time 으로 end_ts 생성 후 diff(초)
# - end_day, station 별로 그룹핑하여 첫 행 op_gap = NULL

# end_ts 만들기 (end_day=YYYYMMDD, end_time=HH:MM:SS.xx 형태 대응)
df["end_day"] = df["end_day"].astype(str)
df["end_time"] = df["end_time"].astype(str)

df["end_ts"] = pd.to_datetime(
    df["end_day"] + " " + df["end_time"],
    errors="coerce"
)

# 파싱 실패 체크
n_bad = int(df["end_ts"].isna().sum())
if n_bad > 0:
    print(f"[WARN] end_ts 파싱 실패 {n_bad}건 (end_time 포맷 확인 필요)")

# 정렬 (요구: end_time 오름차순 + 출력은 end_day,end_time 오름차순)
# 그룹 내 diff 정확도를 위해 end_day, station, end_ts 기준으로 정렬
df = df.sort_values(["end_day", "station", "end_ts", "end_time"], ascending=[True, True, True, True]).reset_index(drop=True)

# 그룹별 op_gap(초) 계산
df["op_gap"] = df.groupby(["end_day", "station"])["end_ts"].diff().dt.total_seconds()

# 첫 행은 NaN -> None (DB에 NULL로)
df["op_gap"] = df["op_gap"].where(df["op_gap"].notna(), None)

# 최종 출력(첨부 이미지 형태)
df_out = df[["end_day","station","end_time","contents","remark","op_gap"]].copy()
df_out.head(20)

,end_day,station,end_time,contents,remark,op_gap
0,20251001,Vision1,01:25:33.71,BA1WJ25273504257SJ8T-14F014-AE :: TEST RESULT ...,PD,NaN
1,20251001,Vision1,01:25:58.21,BA1WJ25273504255SJ8T-14F014-AE :: TEST RESULT ...,PD,24.50
2,20251001,Vision1,01:26:12.13,BA1WJ25273504254SJ8T-14F014-AE :: TEST RESULT ...,PD,13.92
3,20251001,Vision1,01:26:39.74,BA1WJ25273504253SJ8T-14F014-AE :: TEST RESULT ...,PD,27.61
4,20251001,Vision1,01:26:55.06,BA1WJ25273504252SJ8T-14F014-AE :: TEST RESULT ...,PD,15.32
5,20251001,Vision1,01:27:15.24,BA1WJ25273504251SJ8T-14F014-AE :: TEST RESULT ...,PD,20.18
6,20251001,Vision1,01:27:30.52,BA1WJ25273504265SJ8T-14F014-AE :: TEST RESULT ...,PD,15.28
7,20251001,Vision1,01:27:51.61,BA1WJ25273504263SJ8T-14F014-AE :: TEST RESULT ...,PD,21.09
8,20251001,Vision1,01:28:26.05,BA1WJ25273504261SJ8T-14F014-AE :: TEST RESULT ...,PD,34.44
9,20251001,Vision1,01:28:45.27,BA1WJ25273504260SJ8T-14F014-AE :: TEST RESULT ...,PD,19.22


In [6]:
# Cell 6) 타겟 테이블 저장 (기본: 전체 REPLACE 방식)
# - 기존 데이터가 계속 누적되면 중복 가능성이 커서, 우선 "TRUNCATE 후 INSERT"로 구성
# - 만약 누적/UPSERT가 필요하면 알려주면 그 방식으로 바꿔줄게.

with engine.begin() as conn:
    conn.execute(text(f"TRUNCATE TABLE {TGT_SCHEMA}.{TGT_TABLE};"))

# to_sql 저장
df_save = df_out.copy()
df_save.to_sql(
    name=TGT_TABLE,
    con=engine,
    schema=TGT_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print(f"[OK] saved rows = {len(df_save):,} -> {TGT_SCHEMA}.{TGT_TABLE}")

[OK] saved rows = 219,629 -> d1_machine_log.vision_op_gap


In [7]:
# Cell 7) 저장 결과 확인 (샘플 조회)
check_sql = f"""
SELECT id, end_day, station, end_time, contents, remark, op_gap
FROM {TGT_SCHEMA}.{TGT_TABLE}
ORDER BY end_day ASC, end_time ASC
LIMIT 50;
"""
df_chk = pd.read_sql_query(text(check_sql), engine)
df_chk

,id,end_day,station,end_time,contents,remark,op_gap
0,219630,20251001,Vision1,01:25:33.71,BA1WJ25273504257SJ8T-14F014-AE :: TEST RESULT ...,PD,NaN
1,222004,20251001,Vision2,01:25:44.39,BA1WJ25273504236SJ8T-14F014-AE :: TEST RESULT ...,PD,NaN
2,219631,20251001,Vision1,01:25:58.21,BA1WJ25273504255SJ8T-14F014-AE :: TEST RESULT ...,PD,24.50
3,222005,20251001,Vision2,01:25:59.05,BA1WJ25273504235SJ8T-14F014-AE :: TEST RESULT ...,PD,14.66
4,219632,20251001,Vision1,01:26:12.13,BA1WJ25273504254SJ8T-14F014-AE :: TEST RESULT ...,PD,13.92
5,222006,20251001,Vision2,01:26:24.26,BA1WJ25273504234SJ8T-14F014-AE :: TEST RESULT ...,PD,25.21
6,219633,20251001,Vision1,01:26:39.74,BA1WJ25273504253SJ8T-14F014-AE :: TEST RESULT ...,PD,27.61
7,222007,20251001,Vision2,01:26:41.35,BA1WJ25273504233SJ8T-14F014-AE :: TEST RESULT ...,PD,17.09
8,219634,20251001,Vision1,01:26:55.06,BA1WJ25273504252SJ8T-14F014-AE :: TEST RESULT ...,PD,15.32
9,222008,20251001,Vision2,01:26:59.31,BA1WJ25273504232SJ8T-14F014-AE :: TEST RESULT ...,PD,17.96


In [8]:
# Cell 8) station + remark 별 op_gap Boxplot 통계 (소수점 2자리 반올림)

import numpy as np
import pandas as pd
import re

TGT_AV_SCHEMA = "d1_machine_log"
TGT_AV_TABLE  = "vision_op_gap_av"

def r2(x):
    """소수점 3자리에서 반올림 → 2자리"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return round(float(x), 2)

def fmt_num(x: float) -> str:
    """범위 문자열용 포맷 (16.00 → 16.0 유지)"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = f"{x:.2f}".rstrip("0").rstrip(".")
    if re.fullmatch(r"-?\d+", s):
        s = s + ".0"
    return s

def outlier_range_str(vals: pd.Series) -> str:
    if vals is None or len(vals) == 0:
        return ""
    return f"{fmt_num(vals.min())}~{fmt_num(vals.max())}"

# 분석 대상 DF
src = df_out[["station", "remark", "op_gap"]].copy()
src["op_gap"] = pd.to_numeric(src["op_gap"], errors="coerce")
src = src.dropna(subset=["op_gap"]).reset_index(drop=True)

rows = []
for (station, remark), g in src.groupby(["station", "remark"], sort=True):
    s = g["op_gap"]
    n = int(len(s))
    if n == 0:
        continue

    q1 = r2(s.quantile(0.25))
    median = r2(s.quantile(0.50))
    q3 = r2(s.quantile(0.75))
    iqr = q3 - q1

    lower_fence = r2(q1 - 1.5 * iqr)
    upper_fence = r2(q3 + 1.5 * iqr)

    lower_out = s[s < lower_fence]
    upper_out = s[s > upper_fence]
    inlier = s[(s >= lower_fence) & (s <= upper_fence)]

    n_inlier = int(len(inlier))
    del_out_op_gap_av = r2(inlier.mean()) if n_inlier > 0 else None

    rows.append({
        "station": station,
        "remark": remark,
        "op_gap_lower_outlier": outlier_range_str(lower_out),
        "q1": q1,
        "median": median,
        "q3": q3,
        "op_gap_upper_outlier": outlier_range_str(upper_out),
        "del_out_op_gap_av": del_out_op_gap_av,
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "n": n,
        "n_inlier": n_inlier,
    })

df_av = pd.DataFrame(rows)

df_av = df_av[
    [
        "station","remark",
        "op_gap_lower_outlier","q1","median","q3","op_gap_upper_outlier",
        "del_out_op_gap_av",
        "lower_fence","upper_fence","n","n_inlier"
    ]
]

# 출력용 id
df_av_show = df_av.reset_index(drop=True)
df_av_show.insert(0, "id", df_av_show.index + 1)
df_av_show

,id,station,remark,op_gap_lower_outlier,q1,median,q3,op_gap_upper_outlier,del_out_op_gap_av,lower_fence,upper_fence,n,n_inlier
0,1,Vision1,Non-PD,10.07~12.73,14.72,15.07,16.04,18.03~49146.62,15.06,12.74,18.02,19320,16051
1,2,Vision1,PD,,15.80,20.74,25.79,40.78~12980.54,21.66,0.82,40.77,84555,79426
2,3,Vision2,Non-PD,,13.45,15.35,18.21,25.36~49053.11,15.74,6.31,25.35,20555,19342
3,4,Vision2,PD,,14.44,19.68,24.10,38.6~13597.82,19.89,-0.05,38.59,95123,91989


In [9]:
# Cell 9) vision_op_gap_av (remark 포함) 생성 및 저장

from sqlalchemy import text

def q_ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

ddl_av = f"""
CREATE SCHEMA IF NOT EXISTS {q_ident(TGT_AV_SCHEMA)};

CREATE TABLE IF NOT EXISTS {q_ident(TGT_AV_SCHEMA)}.{q_ident(TGT_AV_TABLE)} (
    id BIGSERIAL PRIMARY KEY,
    station TEXT NOT NULL,
    remark TEXT NOT NULL,
    op_gap_lower_outlier TEXT NULL,
    q1 DOUBLE PRECISION NULL,
    median DOUBLE PRECISION NULL,
    q3 DOUBLE PRECISION NULL,
    op_gap_upper_outlier TEXT NULL,
    del_out_op_gap_av DOUBLE PRECISION NULL,
    lower_fence DOUBLE PRECISION NULL,
    upper_fence DOUBLE PRECISION NULL,
    n BIGINT NULL,
    n_inlier BIGINT NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_av))
    conn.execute(text(f"TRUNCATE TABLE {q_ident(TGT_AV_SCHEMA)}.{q_ident(TGT_AV_TABLE)};"))

df_av.to_sql(
    name=TGT_AV_TABLE,
    con=engine,
    schema=TGT_AV_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] saved -> {TGT_AV_SCHEMA}.{TGT_AV_TABLE} (rows={len(df_av):,})")

[OK] saved -> d1_machine_log.vision_op_gap_av (rows=4)


In [10]:
sql_chk = f"""
SELECT
    id, station, remark,
    op_gap_lower_outlier, q1, median, q3, op_gap_upper_outlier,
    del_out_op_gap_av, lower_fence, upper_fence, n, n_inlier
FROM {q_ident(TGT_AV_SCHEMA)}.{q_ident(TGT_AV_TABLE)}
ORDER BY station, remark;
"""
pd.read_sql_query(text(sql_chk), engine)

,id,station,remark,op_gap_lower_outlier,q1,median,q3,op_gap_upper_outlier,del_out_op_gap_av,lower_fence,upper_fence,n,n_inlier
0,9,Vision1,Non-PD,10.07~12.73,14.72,15.07,16.04,18.03~49146.62,15.06,12.74,18.02,19320,16051
1,10,Vision1,PD,,15.80,20.74,25.79,40.78~12980.54,21.66,0.82,40.77,84555,79426
2,11,Vision2,Non-PD,,13.45,15.35,18.21,25.36~49053.11,15.74,6.31,25.35,20555,19342
3,12,Vision2,PD,,14.44,19.68,24.10,38.6~13597.82,19.89,-0.05,38.59,95123,91989


In [11]:
# Cell X) 마지막 DataFrame(df_av)만 원격 DB(100.105.75.47)에 저장

import urllib.parse
from sqlalchemy import create_engine, text

REMOTE_DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TGT_AV_SCHEMA = "d1_machine_log"
TGT_AV_TABLE  = "vision_op_gap_av"

def get_engine_from_cfg(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str, pool_pre_ping=True)

def q_ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

remote_engine = get_engine_from_cfg(REMOTE_DB_CONFIG)

# 1) 스키마/테이블 생성(없으면)
ddl_remote = f"""
CREATE SCHEMA IF NOT EXISTS {q_ident(TGT_AV_SCHEMA)};

CREATE TABLE IF NOT EXISTS {q_ident(TGT_AV_SCHEMA)}.{q_ident(TGT_AV_TABLE)} (
    id BIGSERIAL PRIMARY KEY,
    station TEXT NOT NULL,
    remark TEXT NOT NULL,
    op_gap_lower_outlier TEXT NULL,
    q1 DOUBLE PRECISION NULL,
    median DOUBLE PRECISION NULL,
    q3 DOUBLE PRECISION NULL,
    op_gap_upper_outlier TEXT NULL,
    del_out_op_gap_av DOUBLE PRECISION NULL,
    lower_fence DOUBLE PRECISION NULL,
    upper_fence DOUBLE PRECISION NULL,
    n BIGINT NULL,
    n_inlier BIGINT NULL
);
"""

with remote_engine.begin() as conn:
    conn.execute(text(ddl_remote))
    conn.execute(text(f"TRUNCATE TABLE {q_ident(TGT_AV_SCHEMA)}.{q_ident(TGT_AV_TABLE)};"))

# 2) 저장 (id는 자동 생성되므로 df_av 그대로 insert)
df_av.to_sql(
    name=TGT_AV_TABLE,
    con=remote_engine,
    schema=TGT_AV_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] Remote save complete -> {REMOTE_DB_CONFIG['host']} | {TGT_AV_SCHEMA}.{TGT_AV_TABLE} (rows={len(df_av):,})")

[OK] Remote save complete -> 100.105.75.47 | d1_machine_log.vision_op_gap_av (rows=4)
